# MAIN-1: Ensembl vs CAT Annotation Concordance Sankey

Hierarchical Sankey/alluvial diagram showing progressive concordance between
Ensembl (anchor-sequence projection) and CAT (graph-based projection) annotations.

**Levels:**
1. Gene presence: Present in both | Ensembl only | CAT only
2. RBH status: RBH found | No RBH (subset of 'both')
3. Transcript concordance: Full | Partial | None (subset of 'RBH found')
4. CDS integrity: Intact | Partial | Disrupted (subset of coding RBH)

**Input:** `intermediate_spreadsheets/sankey/` from workflow

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.sankey import Sankey
import numpy as np
from pathlib import Path

# Configuration
RESULTS_DIR = Path('../results')  # Adjust to your pipeline output directory
SANKEY_DIR = RESULTS_DIR / 'intermediate_spreadsheets' / 'sankey'
OUTPUT_DIR = Path('figures')
OUTPUT_DIR.mkdir(exist_ok=True)

# Colour scheme
COLORS = {
    'high_confidence': '#1b4f72',   # Dark blue
    'concordant': '#2e86c1',        # Medium blue
    'partial': '#85c1e9',           # Light blue
    'discordant': '#e74c3c',        # Red
    'ensembl_only': '#f39c12',      # Orange
    'cat_only': '#e67e22',          # Dark orange
    'background': '#ecf0f1',        # Light grey
}

In [ ]:
# Load pre-computed Sankey data
per_asm = pd.read_csv(SANKEY_DIR / 'sankey_per_assembly_flows.tsv', sep='\t')
flow_counts = pd.read_csv(SANKEY_DIR / 'sankey_flow_counts.tsv', sep='\t')

print(f"Loaded data for {len(per_asm)} assemblies")
print(f"\nAggregate flow counts:")
display(flow_counts.T)

In [ ]:
# Compute median flows for the Sankey diagram
medians = {
    # Named genes only (non-ENSG): gene name concordance
    'l1_named_both': per_asm['l1_named_both'].median(),
    'l1_named_ens_only': per_asm['l1_named_ensembl_only'].median(),
    'l1_named_cat_only': per_asm['l1_named_cat_only'].median(),
    # RBH: all gene pairs, own denominator (independent of named gene presence)
    'l2_pass': per_asm['l2_rbh_pass'].median(),
    'l2_fail': per_asm['l2_rbh_fail'].median(),
    'l2_total': per_asm['l2_rbh_total'].median(),
    # Transcript concordance (subset of RBH pass)
    'l3_full': per_asm['l3_tx_full'].median(),
    'l3_partial': per_asm['l3_tx_partial'].median(),
    'l3_none': per_asm['l3_tx_none'].median(),
    # CDS integrity (subset of coding RBH)
    'l4_intact': per_asm['l4_cds_intact'].median(),
    'l4_partial': per_asm['l4_cds_partial'].median(),
    'l4_disrupted': per_asm['l4_cds_disrupted'].median(),
}

# Print headline concordance numbers
named_total = medians['l1_named_both'] + medians['l1_named_ens_only'] + medians['l1_named_cat_only']
print(f"Median named gene concordance: {medians['l1_named_both']/named_total*100:.1f}%  "
      f"(both={int(medians['l1_named_both']):,}, "
      f"ens_only={int(medians['l1_named_ens_only']):,}, "
      f"cat_only={int(medians['l1_named_cat_only']):,})")
print(f"Median RBH pass rate: {per_asm['l2_pct_pass'].median():.1f}%  "
      f"(pass={int(medians['l2_pass']):,} / total={int(medians['l2_total']):,})")
print(f"Median transcript concordance: {per_asm['l3_pct_full'].median():.1f}%")
print(f"Median CDS integrity: {per_asm['l4_pct_intact'].median():.1f}%")

In [ ]:
# --- Agreement-focused stacked bar chart ---
fig, ax = plt.subplots(figsize=(14, 8))

bar_height = 0.6

# Level 1: Named gene presence (non-ENSG only)
named_total = medians['l1_named_both'] + medians['l1_named_ens_only'] + medians['l1_named_cat_only']
l1_vals   = [medians['l1_named_both'], medians['l1_named_ens_only'], medians['l1_named_cat_only']]
l1_pcts   = [v / named_total * 100 for v in l1_vals]
l1_colors = [COLORS['concordant'], COLORS['ensembl_only'], COLORS['cat_only']]
l1_labels = ['Both', 'Ensembl only', 'CAT only']

# Level 2: RBH locus overlap — all gene pairs, own denominator (NOT a subset of L1)
l2_vals   = [medians['l2_pass'], medians['l2_fail']]
l2_pcts   = [v / medians['l2_total'] * 100 if medians['l2_total'] > 0 else 0 for v in l2_vals]
l2_colors = [COLORS['concordant'], COLORS['discordant']]
l2_labels = ['RBH pass', 'No RBH']

# Level 3: Transcript concordance (subset of RBH pass)
l3_vals   = [medians['l3_full'], medians['l3_partial'], medians['l3_none']]
l3_total  = sum(l3_vals)
l3_pcts   = [v / l3_total * 100 if l3_total > 0 else 0 for v in l3_vals]
l3_colors = [COLORS['high_confidence'], COLORS['partial'], COLORS['discordant']]
l3_labels = ['Full match', 'Partial', 'No match']

# Level 4: CDS integrity (subset of coding RBH)
l4_vals   = [medians['l4_intact'], medians['l4_partial'], medians['l4_disrupted']]
l4_total  = sum(l4_vals)
l4_pcts   = [v / l4_total * 100 if l4_total > 0 else 0 for v in l4_vals]
l4_colors = [COLORS['high_confidence'], COLORS['partial'], COLORS['discordant']]
l4_labels = ['Intact', 'Partial disruption', 'Disrupted']

all_levels = [
    (l1_pcts, l1_colors, l1_labels, l1_vals, f'Named genes (n={int(named_total):,})'),
    (l2_pcts, l2_colors, l2_labels, l2_vals, f'All RBH pairs (n={int(medians["l2_total"]):,})'),
    (l3_pcts, l3_colors, l3_labels, l3_vals, f'RBH pass (n={int(l3_total):,})'),
    (l4_pcts, l4_colors, l4_labels, l4_vals, f'Coding RBH (n={int(l4_total):,})'),
]

level_labels = [
    'Named Gene\nPresence',
    'Locus\nOverlap (RBH)',
    'Transcript\nConcordance',
    'CDS\nIntegrity',
]

for i, (pcts, colors, labels, vals, denom_label) in enumerate(all_levels):
    left = 0
    row_y = 3 - i
    for pct, color, label, val in zip(pcts, colors, labels, vals):
        ax.barh(row_y, pct, left=left, height=bar_height, color=color,
                edgecolor='white', linewidth=0.5)
        if pct > 5:
            ax.text(left + pct / 2, row_y,
                    f'{label}\n{pct:.1f}%\n(n={int(val):,})',
                    ha='center', va='center', fontsize=7, fontweight='bold', color='white')
        left += pct
    # Denominator annotation at right margin
    ax.text(102, row_y, denom_label, va='center', ha='left', fontsize=7.5,
            color='#555555', style='italic')

# Divider between rows 1 and 2 to signal they are independent metrics
ax.axhline(y=2.5, color='#aaaaaa', linewidth=0.8, linestyle='--')
ax.text(50, 2.52, 'rows below are subsets of RBH pairs', ha='center',
        fontsize=7, color='#888888', style='italic')

ax.set_yticks(range(4))
ax.set_yticklabels(list(reversed(level_labels)), fontsize=11, fontweight='bold')
ax.set_xlabel('Percentage of row denominator', fontsize=12)
ax.set_xlim(0, 130)
ax.set_title('Ensembl vs CAT Annotation Concordance\n(Median across assemblies)', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure_main1_sankey.png', dpi=300, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figure_main1_sankey.pdf', bbox_inches='tight')
plt.show()
print(f"Saved to {OUTPUT_DIR / 'figure_main1_sankey.png'}")

In [ ]:
# --- Per-assembly distribution strip plot ---
fig, ax = plt.subplots(figsize=(10, 6))

pct_cols = ['l1_named_pct_both', 'l2_pct_pass', 'l3_pct_full', 'l4_pct_intact']
labels = ['Named gene\npresence (both)', 'RBH locus\noverlap', 'Transcript\nconcordance', 'CDS\nintegrity']

for i, (col, label) in enumerate(zip(pct_cols, labels)):
    vals = per_asm[col].dropna()
    jitter = np.random.normal(0, 0.1, len(vals))
    ax.scatter(np.full(len(vals), i) + jitter, vals, alpha=0.3, s=8, color=COLORS['concordant'])
    med = vals.median()
    q25, q75 = vals.quantile(0.25), vals.quantile(0.75)
    ax.plot([i-0.3, i+0.3], [med, med], color='black', linewidth=2)
    ax.plot([i, i], [q25, q75], color='black', linewidth=1.5)

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Percentage (%)', fontsize=12)
ax.set_title('Per-assembly concordance at each level', fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure_main1_per_assembly_distribution.png', dpi=300, bbox_inches='tight')
plt.show()